# Classification d'images CIFAR-10 par réseaux de neurones convolutifs (CNN)

**Architectures VGG · Optimiseurs (SGD / Adam) · Learning rate · Régularisation**

Auteur : Mehdi Redha BELLAHSENE — Méthodologie CRISP-DM — TensorFlow / Keras

---
Ce notebook est **autonome** : il embarque le code des modules du projet et affiche les résultats. Il s'exécute localement ou sur Google Colab.

> Les entraînements complets (grille de 36 configurations, modèle final) sont longs sur CPU. Le notebook **charge et affiche les résultats déjà calculés** (`report_assets/results.json` et figures) lorsqu'ils sont présents, et propose une démonstration d'entraînement rapide.


## 0. Environnement et dépendances


In [ ]:
# Sur Google Colab, décommentez si besoin :
# !pip install -q tensorflow scikit-learn seaborn
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display
import tensorflow as tf
import keras
print('TensorFlow', tf.__version__, '| Keras', keras.__version__)
# --- Accélérateur (Colab) : TPU v5e / GPU T4 / CPU ---
# Exécution > Modifier le type d'exécution > choisir TPU ou GPU.
DEVICE = 'GPU'   # 'GPU' (T4, recommandé) | 'TPU' | 'CPU'
strategy = None
if DEVICE == 'TPU':
    try:
        resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
        tf.config.experimental_connect_to_cluster(resolver)
        tf.tpu.experimental.initialize_tpu_system(resolver)
        strategy = tf.distribute.TPUStrategy(resolver)
        print('TPU prêt :', strategy.num_replicas_in_sync, 'coeur(s)')
    except Exception as e:
        print('TPU indisponible (', e, ') -> CPU/GPU'); strategy = None
elif DEVICE == 'GPU':
    g = tf.config.list_physical_devices('GPU')
    print('GPU :', g if g else 'AUCUN (activez le GPU dans Colab)')
else:
    print('Exécution sur CPU')
os.makedirs('report_assets/curves', exist_ok=True)


## 1. Code du projet (modules embarqués)

Les cellules suivantes définissent le chargement des données, les architectures, l'entraînement et l'évaluation. Elles reprennent le code des modules `src/` du projet.


### 1.1 Données : chargement, normalisation, one-hot, visualisation


In [ ]:
"""
Chargement et préparation du jeu de données CIFAR-10.

Couvre la phase CRISP-DM "Data Understanding & Preparation" :
chargement, exploration, normalisation, one-hot encoding et visualisation.
"""
from __future__ import annotations

import os

import numpy as np
import matplotlib

matplotlib.use("Agg")  # backend non interactif (génération de figures sur disque)
import matplotlib.pyplot as plt
import seaborn as sns

import keras
from keras.utils import to_categorical

# Les 10 classes de CIFAR-10, dans l'ordre des labels (0..9).
CLASS_NAMES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]

# Traductions FR utilisées dans le rapport.
CLASS_NAMES_FR = [
    "avion", "automobile", "oiseau", "chat", "cerf",
    "chien", "grenouille", "cheval", "bateau", "camion",
]


def load_raw():
    """Charge CIFAR-10 brut (valeurs de pixels 0-255, labels entiers)."""
    (x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
    return (x_train, y_train), (x_test, y_test)


def describe(x_train, y_train, x_test, y_test) -> dict:
    """Retourne les caractéristiques structurelles du jeu de données (Tâche 1)."""
    return {
        "train_shape": list(x_train.shape),
        "test_shape": list(x_test.shape),
        "n_train": int(x_train.shape[0]),
        "n_test": int(x_test.shape[0]),
        "image_shape": list(x_train.shape[1:]),
        "n_channels": int(x_train.shape[-1]),
        "n_classes": len(CLASS_NAMES),
    }


def preprocess(x_train, y_train, x_test, y_test):
    """Normalise les pixels dans [0, 1] et applique le one-hot encoding (Tâches 3 & 4)."""
    x_train_norm = x_train.astype("float32") / 255.0
    x_test_norm = x_test.astype("float32") / 255.0
    y_train_oh = to_categorical(y_train, len(CLASS_NAMES))
    y_test_oh = to_categorical(y_test, len(CLASS_NAMES))
    return x_train_norm, y_train_oh, x_test_norm, y_test_oh


def plot_sample_images(x, y, save_path: str, n: int = 10):
    """Affiche n images d'exemple avec leurs labels (Tâche 2)."""
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.figure(figsize=(12, 5))
    for i in range(n):
        plt.subplot(2, 5, i + 1)
        plt.imshow(x[i])
        plt.title(CLASS_NAMES[int(y[i][0])])
        plt.axis("off")
    plt.suptitle("Échantillon d'images CIFAR-10 (32x32, 3 canaux RVB)")
    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.close()


def plot_class_distribution(y_train, save_path: str) -> dict:
    """Trace la distribution des classes et retourne les comptes (Tâche 5)."""
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    unique, counts = np.unique(y_train, return_counts=True)
    plt.figure(figsize=(9, 4.5))
    plt.bar([CLASS_NAMES[int(i)] for i in unique], counts,
            color="#4C72B0", edgecolor="black", linewidth=0.5)
    plt.title("Distribution des classes — jeu d'entraînement CIFAR-10")
    plt.xlabel("Classe")
    plt.ylabel("Nombre d'images")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.close()
    return {CLASS_NAMES[int(i)]: int(c) for i, c in zip(unique, counts)}


### 1.2 Architectures : CNN1 (LeNet-5), VGG1/2/3, variantes régularisées, modèle final


In [ ]:
"""
Définition des architectures CNN.

Modèles conformes à l'énoncé (phase "Data Modeling") :
  - LeNet-5
  - VGG1 / VGG2 / VGG3 : 1, 2 ou 3 blocs VGG.

Spécification de l'énoncé pour un bloc VGG :
  Conv(filters=32, kernel=3x3) -> Conv(filters=32, kernel=3x3) -> MaxPooling(2x2)
  Couche cachée de sortie : Dense(128).

Une variante régularisée (Dropout + BatchNormalization) est fournie pour le bonus.
Un modèle "final" optimisé (capacité accrue + augmentation de données) sert à
maximiser la précision au-delà des contraintes de l'énoncé.
"""
from __future__ import annotations

import keras
from keras import layers, models

NUM_CLASSES = 10
INPUT_SHAPE = (32, 32, 3)

# Paramètres imposés par l'énoncé.
VGG_FILTERS = 32
VGG_KERNEL = (3, 3)
VGG_POOL = (2, 2)
VGG_HIDDEN = 128


def build_lenet5(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES):
    """LeNet-5 (adaptée aux images couleur 32x32).

    Spécification conforme au TP4 du cours : convolutions ReLU avec
    initialisation he_uniform et padding « same » (6 puis 16 filtres 5x5),
    max-pooling 2x2 (strides 2), puis denses 120 -> 84 -> 10.
    """
    return models.Sequential(
        [
            keras.Input(shape=input_shape),
            layers.Conv2D(6, (5, 5), activation="relu",
                          kernel_initializer="he_uniform", padding="same"),
            layers.MaxPooling2D((2, 2), strides=2),
            layers.Conv2D(16, (5, 5), activation="relu",
                          kernel_initializer="he_uniform", padding="same"),
            layers.MaxPooling2D((2, 2), strides=2),
            layers.Flatten(),
            layers.Dense(120, activation="relu"),
            layers.Dense(84, activation="relu"),
            layers.Dense(num_classes, activation="softmax"),
        ],
        name="LeNet5",
    )


def _vgg_block(model, dropout, batchnorm, dropout_rate):
    """Ajoute un bloc VGG : Conv -> Conv -> MaxPool (+ BatchNorm/Dropout en option).

    Dropout et BatchNormalization sont indépendants afin de pouvoir étudier leur
    effet séparément (VGG3 -> VGG3+Drop -> VGG3+Drop+BatchNorm), comme demandé.
    """
    model.add(layers.Conv2D(VGG_FILTERS, VGG_KERNEL, activation="relu", padding="same"))
    if batchnorm:
        model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(VGG_FILTERS, VGG_KERNEL, activation="relu", padding="same"))
    if batchnorm:
        model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D(VGG_POOL))
    if dropout:
        model.add(layers.Dropout(dropout_rate))


def build_vgg(num_blocks: int, dropout: bool = False, batchnorm: bool = False,
              input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES):
    """Construit une architecture VGG-like avec `num_blocks` blocs VGG.

    dropout / batchnorm : activent indépendamment chaque technique de régularisation.
    """
    assert 1 <= num_blocks <= 3, "num_blocks doit être 1, 2 ou 3"
    suffix = ""
    if dropout:
        suffix += "_drop"
    if batchnorm:
        suffix += "_bn"
    model = models.Sequential(name=f"VGG{num_blocks}{suffix}")
    model.add(keras.Input(shape=input_shape))

    # Taux de dropout croissant par profondeur (pratique standard).
    dropout_rates = [0.2, 0.3, 0.4]
    for b in range(num_blocks):
        _vgg_block(model, dropout, batchnorm, dropout_rates[b])

    model.add(layers.Flatten())
    model.add(layers.Dense(VGG_HIDDEN, activation="relu"))
    if batchnorm:
        model.add(layers.BatchNormalization())
    if dropout:
        model.add(layers.Dropout(0.5))
    model.add(layers.Dense(num_classes, activation="softmax"))
    return model


# Fabriques nommées (pratiques pour la boucle de benchmark).
def build_vgg1():
    return build_vgg(1)


def build_vgg2():
    return build_vgg(2)


def build_vgg3():
    return build_vgg(3)


def build_vgg3_drop():
    return build_vgg(3, dropout=True)


def build_vgg3_drop_bn():
    return build_vgg(3, dropout=True, batchnorm=True)


def build_final_model(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES):
    """
    Modèle optimisé pour maximiser la précision (hors contraintes de l'énoncé).

    Différences avec les VGG imposés :
      - augmentation de données intégrée (flip / rotation / zoom) ;
      - nombre de filtres croissant (64 -> 128 -> 256) ;
      - BatchNormalization + Dropout systématiques.
    """
    return models.Sequential(
        [
            keras.Input(shape=input_shape),
            # Augmentation de données (active uniquement à l'entraînement).
            layers.RandomFlip("horizontal"),
            layers.RandomRotation(0.1),
            layers.RandomZoom(0.1),
            # Bloc 1
            layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
            layers.BatchNormalization(),
            layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
            layers.BatchNormalization(),
            layers.MaxPooling2D((2, 2)),
            layers.Dropout(0.2),
            # Bloc 2
            layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
            layers.BatchNormalization(),
            layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
            layers.BatchNormalization(),
            layers.MaxPooling2D((2, 2)),
            layers.Dropout(0.3),
            # Bloc 3
            layers.Conv2D(256, (3, 3), activation="relu", padding="same"),
            layers.BatchNormalization(),
            layers.Conv2D(256, (3, 3), activation="relu", padding="same"),
            layers.BatchNormalization(),
            layers.MaxPooling2D((2, 2)),
            layers.Dropout(0.4),
            # Classifieur
            layers.Flatten(),
            layers.Dense(256, activation="relu"),
            layers.BatchNormalization(),
            layers.Dropout(0.5),
            layers.Dense(num_classes, activation="softmax"),
        ],
        name="FinalVGG",
    )


# Registre des 6 modèles (lignes) du tableau d'évaluation de l'énoncé.
# "CNN1" désigne le CNN de base de l'énoncé, identifié à LeNet-5 (seule
# architecture non-VGG définie dans le sujet).
MODEL_BUILDERS = {
    "CNN1": build_lenet5,
    "VGG1": build_vgg1,
    "VGG2": build_vgg2,
    "VGG3": build_vgg3,
    "VGG3+Drop": build_vgg3_drop,
    "VGG3+Drop+BatchNorm": build_vgg3_drop_bn,
}

# Ordre d'affichage (lignes du tableau).
MODEL_ORDER = list(MODEL_BUILDERS.keys())


### 1.3 Entraînement, callbacks et plot_history()


In [ ]:
"""
Utilitaires d'entraînement et d'évaluation des modèles.

Couvre la phase CRISP-DM "Modeling / Evaluation" : compilation avec un
optimiseur donné (SGD ou Adam), entraînement avec callbacks, et la fonction
`plot_history` demandée par l'énoncé.
"""
from __future__ import annotations

import os

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

import keras


def make_optimizer(name: str, learning_rate: float | None = None):
    """Instancie l'optimiseur demandé (SGD avec momentum, ou Adam)."""
    name = name.upper()
    if name == "SGD":
        return keras.optimizers.SGD(learning_rate=learning_rate or 0.01, momentum=0.9)
    if name == "ADAM":
        return keras.optimizers.Adam(learning_rate=learning_rate or 0.001)
    raise ValueError(f"Optimiseur inconnu : {name}")


def make_callbacks(patience: int = 6, monitor: str = "val_accuracy"):
    """EarlyStopping + réduction du learning rate sur plateau."""
    return [
        keras.callbacks.EarlyStopping(
            monitor=monitor, patience=patience, mode="max",
            restore_best_weights=True, verbose=0,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5, verbose=0,
        ),
    ]


def compile_model(model, optimizer_name: str, learning_rate: float | None = None):
    model.compile(
        optimizer=make_optimizer(optimizer_name, learning_rate),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def train_and_evaluate(build_fn, optimizer_name, learning_rate, data, model_label,
                       epochs=50, batch_size=64, callbacks=None, verbose=0, strategy=None):
    """
    Construit, entraîne et évalue un modèle pour un couple (optimiseur, learning rate).

    Retourne un dictionnaire sérialisable + l'objet history Keras pour le tracé.
    Les métriques d'entraînement ET de test sont conservées (le tableau de l'énoncé
    demande l'accuracy et la perte sur les deux ensembles).

    `strategy` : stratégie tf.distribute optionnelle (ex. TPUStrategy sur Colab).
    Si fournie, le modèle est construit et compilé dans son scope. Par défaut (None),
    comportement local CPU/GPU inchangé.
    """
    x_train, y_train, x_test, y_test = data
    if strategy is not None:
        with strategy.scope():
            model = build_fn()
            compile_model(model, optimizer_name, learning_rate)
    else:
        model = build_fn()
        compile_model(model, optimizer_name, learning_rate)

    history = model.fit(
        x_train, y_train,
        epochs=epochs, batch_size=batch_size,
        validation_data=(x_test, y_test),
        callbacks=callbacks if callbacks is not None else make_callbacks(),
        verbose=verbose,
    )

    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    train_loss, train_acc = model.evaluate(x_train, y_train, verbose=0)
    result = {
        "model": model_label,
        "arch_name": model.name,
        "optimizer": optimizer_name,
        "learning_rate": learning_rate,
        "test_accuracy": float(test_acc),
        "test_loss": float(test_loss),
        "train_accuracy": float(train_acc),
        "train_loss": float(train_loss),
        "overfit_gap": float(train_acc - test_acc),
        "epochs_run": len(history.history["loss"]),
        "n_params": int(model.count_params()),
        "best_val_accuracy": float(max(history.history["val_accuracy"])),
    }
    return result, history, model


def plot_history(history, title: str, save_path: str):
    """
    Trace l'évolution de la perte et de l'accuracy (entraînement + validation).

    C'est la fonction `plot_history(history)` demandée dans l'énoncé (section 2.3).
    """
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    h = history.history
    # Courbes en couleur : entraînement (trait plein) vs validation (pointillé).
    plt.figure(figsize=(12, 4.5))

    plt.subplot(1, 2, 1)
    plt.plot(h["loss"], color="#1f77b4", linestyle="-", label="Entraînement")
    plt.plot(h["val_loss"], color="#d62728", linestyle="--", label="Validation (test)")
    plt.title(f"{title} — Perte (loss)")
    plt.xlabel("Époque")
    plt.ylabel("Perte")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(h["accuracy"], color="#1f77b4", linestyle="-", label="Entraînement")
    plt.plot(h["val_accuracy"], color="#d62728", linestyle="--", label="Validation (test)")
    plt.title(f"{title} — Précision (accuracy)")
    plt.xlabel("Époque")
    plt.ylabel("Précision")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.close()


### 1.4 Évaluation : matrice de confusion, précision par classe, comparaisons


In [ ]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

def predict_classes(model, x_test):
    probs = model.predict(x_test, verbose=0)
    return np.argmax(probs, axis=1), probs


def plot_confusion_matrix(y_true, y_pred, save_path):
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    cm = confusion_matrix(y_true, y_pred)
    cm_n = cm.astype("float") / cm.sum(axis=1, keepdims=True) * 100
    plt.figure(figsize=(9, 7.5))
    sns.heatmap(cm_n, annot=True, fmt=".0f", cmap="Blues", linewidths=0.5,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cbar_kws={"label": "%"})
    plt.title("Matrice de confusion (% par classe réelle)")
    plt.xlabel("Classe prédite"); plt.ylabel("Classe réelle")
    plt.tight_layout(); plt.savefig(save_path, dpi=120, bbox_inches="tight"); plt.close()
    return cm


def per_class_accuracy(y_true, y_pred, save_path) -> dict:
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    cm = confusion_matrix(y_true, y_pred)
    acc = cm.diagonal() / cm.sum(axis=1)
    order = np.argsort(acc)
    plt.figure(figsize=(9, 4.5))
    colors = ["#C44E52" if a < acc.mean() else "#55A868" for a in acc[order]]
    plt.bar([CLASS_NAMES[i] for i in order], acc[order] * 100, color=colors,
            edgecolor="black", linewidth=0.4)
    plt.axhline(acc.mean() * 100, color="grey", linestyle="--",
                label=f"Moyenne = {acc.mean()*100:.1f}%")
    plt.title("Précision par classe"); plt.xlabel("Classe"); plt.ylabel("Précision (%)")
    plt.xticks(rotation=45); plt.legend()
    plt.tight_layout(); plt.savefig(save_path, dpi=120, bbox_inches="tight"); plt.close()
    return {CLASS_NAMES[i]: float(acc[i]) for i in range(len(CLASS_NAMES))}


def plot_misclassified(model, x_test, y_true, save_path, n=10):
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    y_pred, _ = predict_classes(model, x_test)
    wrong = np.where(y_pred != y_true)[0][:n]
    plt.figure(figsize=(12, 5))
    for i, idx in enumerate(wrong):
        plt.subplot(2, 5, i + 1)
        plt.imshow(x_test[idx])
        plt.title(f"V: {CLASS_NAMES[int(y_true[idx])]}\nP: {CLASS_NAMES[int(y_pred[idx])]}", fontsize=9)
        plt.axis("off")
    plt.suptitle("Exemples mal classés (V = réel, P = prédit)")
    plt.tight_layout(); plt.savefig(save_path, dpi=120, bbox_inches="tight"); plt.close()


def _short(m):
    """Nom court d'architecture pour les figures."""
    return m.replace("+Drop+BatchNorm", "+D+BN").replace("+Drop", "+D")


def plot_model_comparison(results, save_path, model_order, lrs, opts):
    """Heatmap annotée ; les cellules en divergence (<=15 %, ~hasard) sont hachurées."""
    from matplotlib.patches import Rectangle
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    lookup = {(r["model"], r["learning_rate"], r["optimizer"]): r["test_accuracy"] for r in results}
    cols = [f"LR={lr}\n{o}" for lr in lrs for o in opts]
    mat = np.full((len(model_order), len(cols)), np.nan)
    for mi, m in enumerate(model_order):
        ci = 0
        for lr in lrs:
            for o in opts:
                v = lookup.get((m, lr, o))
                if v is not None:
                    mat[mi, ci] = v * 100
                ci += 1
    plt.figure(figsize=(11, 6))
    ax = sns.heatmap(mat, annot=True, fmt=".1f", cmap="RdYlGn", vmin=10, vmax=95,
                     xticklabels=cols, yticklabels=[_short(m) for m in model_order],
                     cbar_kws={"label": "Précision test (%)"}, linewidths=0.6, linecolor="white",
                     annot_kws={"fontsize": 9, "fontweight": "bold"})
    for mi in range(mat.shape[0]):
        for ci in range(mat.shape[1]):
            if not np.isnan(mat[mi, ci]) and mat[mi, ci] <= 15:
                ax.add_patch(Rectangle((ci, mi), 1, 1, fill=False, edgecolor="black", lw=1.4, hatch="///"))
    plt.title("Précision de test (%) par modèle, learning rate et optimiseur\n"
              "(hachuré = divergence, ~10 % = niveau du hasard)", fontsize=11)
    plt.ylabel("Modèle"); plt.xlabel("")
    plt.tight_layout(); plt.savefig(save_path, dpi=130, bbox_inches="tight"); plt.close()


def plot_lr_sensitivity(results, save_path, model_order, lrs):
    """Précision vs learning rate (échelle log), par optimiseur, avec niveau du hasard."""
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    opts = sorted({r["optimizer"] for r in results})
    lookup = {(r["model"], r["learning_rate"], r["optimizer"]): r["test_accuracy"] for r in results}
    markers = ["o", "s", "^", "D", "v", "P"]
    fig, axes = plt.subplots(1, len(opts), figsize=(13, 5.5), sharey=True)
    if len(opts) == 1:
        axes = [axes]
    for ax, o in zip(axes, opts):
        for k, m in enumerate(model_order):
            ys = [lookup.get((m, lr, o)) * 100 if lookup.get((m, lr, o)) is not None else np.nan
                  for lr in lrs]
            ax.plot(lrs, ys, marker=markers[k % len(markers)], markersize=7, linewidth=1.8, label=_short(m))
        ax.set_xscale("log"); ax.set_xticks(lrs); ax.set_xticklabels([str(l) for l in lrs])
        ax.axhspan(0, 15, color="red", alpha=0.06)
        ax.axhline(10, color="grey", ls=":", lw=1.2)
        ax.text(lrs[0], 12, "niveau du hasard (10 %)", fontsize=7.5, color="grey")
        ax.set_title(f"Optimiseur : {o}", fontsize=11); ax.set_xlabel("Learning rate (échelle log)")
        ax.grid(True, alpha=0.3); ax.set_ylim(0, 100)
    axes[0].set_ylabel("Précision de test (%)")
    axes[-1].legend(fontsize=8, title="Modèle", loc="lower left")
    plt.suptitle("Sensibilité de la précision au learning rate", fontsize=12)
    plt.tight_layout(); plt.savefig(save_path, dpi=130, bbox_inches="tight"); plt.close()


def plot_accuracy_bars(results, save_path, model_order, best_lr, opts):
    """Barres groupées : précision de test par architecture, un jeu de barres par optimiseur."""
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    lookup = {(r["model"], r["learning_rate"], r["optimizer"]): r["test_accuracy"] for r in results}
    x = np.arange(len(model_order)); w = 0.8 / max(1, len(opts))
    palette = {"SGD": "#4C72B0", "Adam": "#DD8452"}
    plt.figure(figsize=(11, 5.5))
    for k, o in enumerate(opts):
        vals = [(lookup.get((m, best_lr, o)) or 0) * 100 for m in model_order]
        xb = x + (k - (len(opts) - 1) / 2) * w
        bars = plt.bar(xb, vals, w, label=o, color=palette.get(o), edgecolor="black", linewidth=0.4)
        for b, v in zip(bars, vals):
            plt.text(b.get_x() + b.get_width() / 2, v + 0.8, f"{v:.1f}", ha="center", fontsize=7.5)
    plt.xticks(x, [_short(m) for m in model_order], rotation=20, ha="right")
    plt.ylabel("Précision de test (%)"); plt.ylim(0, 100)
    plt.title(f"Précision de test par architecture et optimiseur (learning rate = {best_lr})")
    plt.legend(title="Optimiseur"); plt.grid(axis="y", alpha=0.3)
    plt.tight_layout(); plt.savefig(save_path, dpi=130, bbox_inches="tight"); plt.close()


def plot_train_test_gap(results, save_path, model_order, best_lr, opt):
    """Barres entraînement vs test (au meilleur LR + optimiseur) -> sur-apprentissage."""
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    lut = {(r["model"], r["learning_rate"], r["optimizer"]): r for r in results}
    rows = [lut.get((m, best_lr, opt)) for m in model_order]
    if not any(r and r.get("train_accuracy") is not None for r in rows):
        return
    tr = [(r.get("train_accuracy") or 0) * 100 if r else 0 for r in rows]
    te = [(r.get("test_accuracy") or 0) * 100 if r else 0 for r in rows]
    x = np.arange(len(model_order)); w = 0.38
    plt.figure(figsize=(11, 5.5))
    plt.bar(x - w / 2, tr, w, label="Entraînement", color="#55A868", edgecolor="black", linewidth=0.4)
    plt.bar(x + w / 2, te, w, label="Test", color="#C44E52", edgecolor="black", linewidth=0.4)
    for i in range(len(model_order)):
        plt.text(x[i], max(tr[i], te[i]) + 1.2, f"écart {tr[i]-te[i]:.0f}", ha="center", fontsize=7.5)
    plt.xticks(x, [_short(m) for m in model_order], rotation=20, ha="right")
    plt.ylabel("Précision (%)"); plt.ylim(0, 108)
    plt.title(f"Entraînement vs test ({opt}, LR={best_lr}) — l'écart mesure le sur-apprentissage")
    plt.legend(); plt.grid(axis="y", alpha=0.3)
    plt.tight_layout(); plt.savefig(save_path, dpi=130, bbox_inches="tight"); plt.close()


## 2. Compréhension et préparation des données (CRISP-DM)

Chargement de CIFAR-10, exploration (dimensions, échantillons, distribution des classes), normalisation des pixels dans [0,1] et encodage one-hot des labels.


In [ ]:
(x_train, y_train), (x_test, y_test) = load_raw()
info = describe(x_train, y_train, x_test, y_test)
for k, v in info.items():
    print(f'{k:14s}: {v}')

plot_sample_images(x_train, y_train, 'report_assets/samples.png')
plot_class_distribution(y_train, 'report_assets/class_distribution.png')
display(Image('report_assets/samples.png'))
display(Image('report_assets/class_distribution.png'))

x_train_n, y_train_oh, x_test_n, y_test_oh = preprocess(x_train, y_train, x_test, y_test)
print('Pixels normalisés dans [', x_train_n.min(), ',', x_train_n.max(), ']')
print('Labels one-hot, exemple :', y_train_oh[0])


## 3. Architectures

Bloc VGG = Conv(32, 3x3) → Conv(32, 3x3) → MaxPool(2x2), couche cachée Dense(128), sortie softmax(10). On compare 6 modèles : CNN1 (LeNet-5), VGG1/2/3, VGG3+Dropout et VGG3+Dropout+BatchNorm.


In [ ]:
for name in MODEL_ORDER:
    m = MODEL_BUILDERS[name]()
    print(f'{name:22s} {m.count_params():>9,} paramètres')

# Résumé détaillé d'un modèle (ex. VGG3)
build_vgg3().summary()


## 4. Démonstration d'entraînement (rapide)

Pour illustrer la boucle d'entraînement et `plot_history`, on entraîne un modèle sur quelques époques. **La grille complète** (36 configurations : 6 modèles × 3 learning rates × 2 optimiseurs, MaxEpochs=50) est produite par `run_benchmarks.py` — voir la section 5 pour les résultats complets.


In [ ]:
QUICK_DEMO = True   # passez à False pour sauter la démo
if QUICK_DEMO:
    data = (x_train_n, y_train_oh, x_test_n, y_test_oh)
    res, hist, _ = train_and_evaluate(
        build_vgg2, 'Adam', 0.001, data, 'VGG2',
        epochs=5, callbacks=make_callbacks(patience=3))
    print('Démo VGG2/Adam/LR=0.001 :', res['test_accuracy'])
    plot_history(hist, 'Démo VGG2 / Adam', 'report_assets/demo_curve.png')
    display(Image('report_assets/demo_curve.png'))


## 4 bis. Entraînement complet sur GPU (Colab) — optionnel

Sur un GPU (Colab T4, ~16 Go), la grille complète (36 configurations) puis le modèle final s'entraînent en quelques dizaines de minutes. Passez `RUN_FULL` à `True`, vérifiez que le GPU est actif (cellule 0), puis exécutez. Les résultats et figures (en noir et blanc) sont sauvegardés dans `report_assets/` et affichés à la section 5.


In [ ]:
# Auto : True si un accélérateur (GPU/TPU) est détecté ; sinon False (évite ~8h sur CPU).
# Forcez la valeur si besoin : RUN_FULL = True  /  RUN_FULL = False
RUN_FULL = bool(tf.config.list_physical_devices('GPU')) or (strategy is not None)
print('RUN_FULL =', RUN_FULL, '(entraînement complet)' if RUN_FULL else '(grille sautée — pas d accélérateur)')
if RUN_FULL:
    import json, time
    LRS = [0.001, 0.01, 0.1]; OPTS = ['SGD', 'Adam']
    data = (x_train_n, y_train_oh, x_test_n, y_test_oh)
    cd = plot_class_distribution(y_train, 'report_assets/class_distribution.png')
    results = []
    for lr in LRS:
        for name in MODEL_ORDER:
            for opt in OPTS:
                cb = [keras.callbacks.EarlyStopping(monitor='val_accuracy',
                      patience=8, mode='max', restore_best_weights=True)]
                res, hist, _ = train_and_evaluate(MODEL_BUILDERS[name], opt, lr,
                                                  data, name, epochs=50,
                                                  callbacks=cb, strategy=strategy)
                slug = name.lower().replace('+','_').replace('','')
                cpath = f'report_assets/curves/{slug}_lr{str(lr).replace(".","_")}_{opt.lower()}.png'
                plot_history(hist, f'{name} / {opt} / LR={lr}', cpath)
                res['curve'] = os.path.relpath(cpath, 'report_assets')
                results.append(res)
                print(f"{name:20s} {opt:4s} lr={lr:<6} -> test={res['test_accuracy']:.3f}")
    payload = {'dataset': {**describe(x_train, y_train, x_test, y_test),
                           'class_distribution': cd},
               'benchmark': results,
               'config': {'max_epochs': 50, 'batch_size': 64, 'learning_rates': LRS,
                          'optimizers': OPTS, 'model_order': MODEL_ORDER}}
    json.dump(payload, open('report_assets/results.json', 'w', encoding='utf-8'),
              indent=2, ensure_ascii=False)
    plot_model_comparison(results, 'report_assets/comparison.png', MODEL_ORDER, LRS, OPTS)
    plot_lr_sensitivity(results, 'report_assets/lr_sensitivity.png', MODEL_ORDER, LRS)
    _means = {lr: float(np.mean([r['test_accuracy'] for r in results if r['learning_rate']==lr])) for lr in LRS}
    _blr = max(_means, key=_means.get)
    plot_accuracy_bars(results, 'report_assets/accuracy_bars.png', MODEL_ORDER, _blr, OPTS)
    plot_train_test_gap(results, 'report_assets/train_test_gap.png', MODEL_ORDER, _blr, 'Adam')
    print('Grille terminée :', len(results), 'configurations.')


### Modèle retenu (conforme à l'énoncé)

La meilleure configuration de la grille est réentraînée plus longtemps (n'utilise que les architectures imposées). C'est le modèle retenu du rapport.


In [ ]:
if RUN_FULL:
    import json
    best = max(results, key=lambda r: r['test_accuracy'])
    print('Meilleure config :', best['model'], best['optimizer'], best['learning_rate'])
    if strategy:
        with strategy.scope():
            rmod = MODEL_BUILDERS[best['model']](); compile_model(rmod, best['optimizer'], best['learning_rate'])
    else:
        rmod = MODEL_BUILDERS[best['model']](); compile_model(rmod, best['optimizer'], best['learning_rate'])
    cbr = [keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=15, mode='max', restore_best_weights=True),
           keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)]
    hr = rmod.fit(x_train_n, y_train_oh, epochs=100, batch_size=64,
                  validation_data=(x_test_n, y_test_oh), callbacks=cbr, verbose=2)
    plot_history(hr, f"Modèle retenu ({best['model']})", 'report_assets/retained_curves.png')
    yp, _ = predict_classes(rmod, x_test_n); yt = y_test.flatten()
    plot_confusion_matrix(yt, yp, 'report_assets/retained_confusion.png')
    pcr = per_class_accuracy(yt, yp, 'report_assets/retained_per_class.png')
    rl, ra = rmod.evaluate(x_test_n, y_test_oh, verbose=0)
    d = json.load(open('report_assets/results.json', encoding='utf-8'))
    d['retained_model'] = {'label': best['model'], 'optimizer': best['optimizer'],
        'learning_rate': best['learning_rate'], 'test_accuracy': float(ra), 'test_loss': float(rl),
        'n_params': int(rmod.count_params()), 'epochs_run': len(hr.history['loss']),
        'epochs_max': 100, 'per_class_accuracy': pcr}
    json.dump(d, open('report_assets/results.json', 'w', encoding='utf-8'), indent=2, ensure_ascii=False)
    print('Modèle retenu — précision test :', round(ra, 4))


### Bonus 1 — Modèle optimisé *from-scratch* (sans poids pré-entraînés)

Premier modèle « pour aller plus loin », entraîné de zéro : réseau plus profond (filtres 64→128→256, BatchNorm, Dropout) + augmentation de données. Vise ~89 %.


In [ ]:
if RUN_FULL:
    # (1) Modèle optimisé FROM-SCRATCH (entrée normalisée [0,1])
    fm = build_final_model()
    compile_model(fm, 'Adam', 1e-3)
    cb = [keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=12, mode='max',
                                        restore_best_weights=True),
          keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6)]
    h = fm.fit(x_train_n, y_train_oh, epochs=60, batch_size=64,
               validation_data=(x_test_n, y_test_oh), callbacks=cb, verbose=2)
    plot_history(h, 'Modele optimise (from-scratch)', 'report_assets/final_curves.png')
    yp, _ = predict_classes(fm, x_test_n); yt = y_test.flatten()
    plot_confusion_matrix(yt, yp, 'report_assets/confusion_matrix.png')
    pcr = per_class_accuracy(yt, yp, 'report_assets/per_class_accuracy.png')
    plot_misclassified(fm, x_test_n, yt, 'report_assets/misclassified.png')
    import json
    rl, ra = fm.evaluate(x_test_n, y_test_oh, verbose=0)
    d = json.load(open('report_assets/results.json', encoding='utf-8'))
    d['final_model'] = {'name': 'FinalVGG (from-scratch)', 'test_accuracy': float(ra),
        'test_loss': float(rl), 'n_params': int(fm.count_params()),
        'epochs_run': len(h.history['loss']), 'epochs_max': 60, 'per_class_accuracy': pcr}
    json.dump(d, open('report_assets/results.json', 'w', encoding='utf-8'), indent=2, ensure_ascii=False)
    print('From-scratch -> precision test :', round(ra, 4))


### Bonus 2 — Transfer learning EfficientNetB5 (exploite le GPU, précision maximale)

Second modèle : **EfficientNetB5 pré-entraîné ImageNet**, images agrandies 32→384, fine-tuning 2 phases en **mixed precision** (Tensor Cores). Vise ~97-98 %. *(`@456` = max absolu mais ~4-8 h + risque OOM. Si OOM : `BATCH=32`.)*


In [ ]:
if RUN_FULL:
    # (2) Transfer learning EfficientNetB5 -- exploite le GPU (mixed precision). Entrée BRUTE [0,255].
    import json
    BACKBONE, IMG, BATCH = 'B5', 456, 64   # B5@456 = resolution native (precision max) ; GPU 80GB : tu peux monter BATCH a 96/128
    keras.mixed_precision.set_global_policy('mixed_float16')
    APP = {'B0': keras.applications.EfficientNetB0,
           'B3': keras.applications.EfficientNetB3,
           'B5': keras.applications.EfficientNetB5}[BACKBONE]
    base = APP(include_top=False, weights='imagenet', input_shape=(IMG, IMG, 3))
    base.trainable = False
    inp = keras.Input(shape=(32, 32, 3))
    z = keras.layers.Resizing(IMG, IMG)(inp)
    z = keras.layers.RandomFlip('horizontal')(z)
    z = keras.layers.RandomRotation(0.05)(z)
    z = base(z, training=False)
    z = keras.layers.GlobalAveragePooling2D()(z)
    z = keras.layers.Dropout(0.3)(z)
    out = keras.layers.Dense(10, activation='softmax', dtype='float32')(z)  # float32 (mixed precision)
    tmodel = keras.Model(inp, out, name=f'EfficientNet{BACKBONE}_TL')

    def _cc(m, lr):
        opt = keras.mixed_precision.LossScaleOptimizer(keras.optimizers.Adam(lr))
        m.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])

    _cc(tmodel, 1e-3)
    print('Transfer B5 - phase 1 (tete)...')
    tmodel.fit(x_train, y_train_oh, epochs=3, batch_size=BATCH,
               validation_data=(x_test, y_test_oh), verbose=1)
    base.trainable = True
    _cc(tmodel, 1e-5)
    cb = [keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5, mode='max',
                                        restore_best_weights=True),
          keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7)]
    print('Transfer B5 - phase 2 (fine-tuning)...')
    h = tmodel.fit(x_train, y_train_oh, epochs=15, batch_size=BATCH,
                   validation_data=(x_test, y_test_oh), callbacks=cb, verbose=1)
    plot_history(h, f'Transfer learning ({tmodel.name})', 'report_assets/transfer_curves.png')
    yp, _ = predict_classes(tmodel, x_test); yt = y_test.flatten()
    plot_confusion_matrix(yt, yp, 'report_assets/transfer_confusion.png')
    pcr = per_class_accuracy(yt, yp, 'report_assets/transfer_per_class.png')
    plot_misclassified(tmodel, x_test, yt, 'report_assets/transfer_misclassified.png')
    rl, ra = tmodel.evaluate(x_test, y_test_oh, verbose=0)
    d = json.load(open('report_assets/results.json', encoding='utf-8'))
    d['transfer_model'] = {'name': f'EfficientNet{BACKBONE}_TL', 'test_accuracy': float(ra),
        'test_loss': float(rl), 'n_params': int(tmodel.count_params()),
        'epochs_run': len(h.history['loss']), 'epochs_max': 15, 'img_size': IMG, 'per_class_accuracy': pcr}
    json.dump(d, open('report_assets/results.json', 'w', encoding='utf-8'), indent=2, ensure_ascii=False)
    tmodel.save('report_assets/transfer_model.keras')
    keras.mixed_precision.set_global_policy('float32')
    print('Transfer B5 -> precision test :', round(ra, 4))


## 5. Résultats complets de la grille (chargés)

On charge `report_assets/results.json` (produit par `run_benchmarks.py`) et on affiche le tableau des performances ainsi que les figures de synthèse.


In [ ]:
import json, pandas as pd
path = 'report_assets/results.json'
if os.path.exists(path):
    d = json.load(open(path, encoding='utf-8'))
    b = d['benchmark']
    df = pd.DataFrame([{ 'Modèle': r['model'], 'LR': r['learning_rate'],
                         'Optimiseur': r['optimizer'],
                         'Acc. test (%)': round(r['test_accuracy']*100, 2),
                         'Acc. train (%)': round(r['train_accuracy']*100, 2),
                         'Perte test': round(r['test_loss'], 3),
                         'Époques': r['epochs_run'] } for r in b])
    # Tableau pivot : modèles x (LR, optimiseur) sur la précision de test
    pivot = df.pivot_table(index='Modèle', columns=['LR', 'Optimiseur'],
                           values='Acc. test (%)')
    display(pivot)
    print(f"{len(b)}/36 configurations calculées.")
    for fig in ['comparison.png', 'lr_sensitivity.png']:
        p = f'report_assets/{fig}'
        if os.path.exists(p):
            display(Image(p))
else:
    print('Aucun results.json. Sur Colab : mettez RUN_FULL=True (section 4 bis) '
          'et exécutez. En local : python run_benchmarks.py --epochs 50.')


## 7. Conclusion

- Le prétraitement (normalisation, one-hot) est indispensable à un entraînement stable.
- La profondeur (CNN1 → VGG3) améliore l'extraction de caractéristiques.
- **Adam** converge plus vite et tolère mieux le learning rate que **SGD**.
- Un **learning rate** trop élevé (0.1) déstabilise l'entraînement.
- **Dropout** et **Batch Normalization** réduisent le surapprentissage.

Le rapport PDF détaillé (`report/Rapport_ML_CIFAR10.pdf`) présente l'analyse complète et les résultats chiffrés.


## 8. Mise à l'épreuve du modèle — ML Playground

Le modèle de transfer learning est déployé dans **ML Playground** (https://machine-learning.bellahsene.org), une application web et mobile développée pour le projet : trois mini-jeux (**Le Duel** humain vs machine, **Dessine je devine**, **Test Ultime CINIC-10** sur des images jamais vues) confrontent le modèle à des utilisateurs réels. Voir la section 8 du rapport et le dossier `playground-cifar-model/`.
